# Datathon Fase 5 — Passos Mágicos
## Notebook 01 · Limpeza e Preparação da Base (PEDE 2022–2024)

**Filosofia deste notebook:**
1. **O documento `PEDE_Pontos importantes` é a primeira fonte de verdade.** Toda regra de tratamento
   parte do que o documento **declara** (fórmulas, faixas, pesos). A base serve para **verificar
   conformidade** — quando o dado viola a definição, é o dado que está errado, não a regra.
2. **Todas as decisões são parâmetros ligáveis** (célula "PAINEL DE DECISÕES"). Você liga/desliga
   cada tratamento e vê o impacto. Nada de regra escondida no meio do código.
3. **Rastreabilidade:** ao final, uma tabela mostra quantas linhas cada decisão afetou.

> Ajuste o `CAMINHO_BASE` e o **PAINEL DE DECISÕES**. Rode em ordem.


## ⚙️ Configuração e PAINEL DE DECISÕES

Cada parâmetro abaixo é uma decisão de limpeza. O comentário cita a base documental.
Mude para testar cenários — a tabela final mostra o efeito de cada um.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CAMINHO_BASE = '/content/drive/MyDrive/Datathon/BASE_DE_DADOS_PEDE_2024_-_DATATHON_original.xlsx'
    PASTA_SAIDA  = '/content/drive/MyDrive/Datathon/'
except ImportError:                      # execução fora do Colab
    CAMINHO_BASE = 'BASE_DE_DADOS_PEDE_2024_-_DATATHON_original.xlsx'
    PASTA_SAIDA  = './saida/'

# ══════════════ PAINEL DE DECISÕES ══════════════
# IDA — Documento: "IDA = (Nota Mat + Nota Por + Nota Ing) / 3"
RECALCULAR_IDA          = True    # recalcula IDA em vez de usar o da base (que mistura arredondamentos)?
IDA_MEDIA_NOTAS_PRESENTES = True  # True: média das notas presentes (trata Ing ausente nas Fases 0-2).
                                  # False: exige as 3 notas (Mat,Por,Ing), senão NaN.

# IAA — Tabela 40 + Figura 10: escala com piso por faixa de fase (0-2: 3 opções, piso 3,5;
# 3-8: 4 opções, piso 2,5). Valor abaixo do piso é impossível pela escala => inválido.
# O INDE da ONG que bate com a fórmula usando o IAA inválido está contaminado por ele.
PISO_IAA_FASE_0_2 = 3.5
PISO_IAA_FASE_3_8 = 2.5
TOL_CONTAMINACAO  = 0.02          # |INDE_base - fórmula(IAA inválido)| <= tol => contaminação provada
# IPS/IPP/IPV — sem tabela de pontuação no documento; piso ~2,5 é EMPÍRICO (nenhum valor
# entre 0 e 2,5 na base). Zero tratado como ausência por esse fundamento, não pela Tabela 40.
TRATAR_ZEROS_IMPOSSIVEIS = True   # converte 0 -> NaN em IPS/IPP/IPV?

# IEG — Documento: engajamento em tarefas. 0 real é raro; tratamos como ausência só no ciclo vazio.
TRATAR_IEG_ZERO_NO_CICLO_VAZIO = True

# IPP 2022 — não veio como coluna, mas COMPÕE o INDE 2022 (peso 0,1 nas Fases 0-7).
RECONSTRUIR_IPP_2022     = True    # recupera IPP via engenharia reversa do INDE?
CLIP_INDICES_FORA_FAIXA  = True    # valores fora de [0,10] (ex: -0,07 do IPP reconstruído) -> NaN?
IPP_COMO_FEATURE         = True    # IPP entra como feature (reconstrução algébrica exata; sensibilidade no NB02)

# Idade — convenção de corte. Documento (Tabela 4) associa fase a idade ideal.
CONVENCAO_IDADE = '31-12'         # '31-12' (idade no ano civil) — padrão. Alternativa conceitual: '31-03'.

# Fase efetiva — progressão registrada pela ONG
APLICAR_CORRECAO_FASE     = True      # corrige fase além do ideal (salto>=2 e fase>ideal): fase_ant+1, iterativo
TRATAMENTO_SALTOS_COERENTES = 'excluir'  # 'excluir' | 'manter' — saltos >=2 rumo ao ideal e regressões
                                         # (mecânica não documentada; decisão conservadora, reavaliar pós-plantão)

# Exclusão de linhas do modelo
CORTE_MIN_FEATURES_MODELO = 3     # mínimo de features para manter a linha; abaixo disso, exclui.
ZERO_IMPUTACAO            = True  # True: qualquer feature ausente (pós-álgebra) exclui a linha (SEM KNN).
EXCLUIR_FASE_8_DO_MODELO  = True  # universitários: fórmula de INDE diferente + sem coleta de ciclo.
EXCLUIR_FASE_9_DE_TUDO    = True  # matrícula em processamento ("INCLUIR"): dado fabricado.
# ═════════════════════════════════════════════════

import pandas as pd, numpy as np, re
pd.set_option('display.max_columns', None)
log_impacto = {}   # registra o efeito de cada decisão


In [ ]:
sheets = pd.read_excel(CAMINHO_BASE, sheet_name=None)
s22 = sheets['PEDE2022'].copy()
s23 = sheets['PEDE2023'].copy()
s24 = sheets['PEDE2024'].copy()
print('Abas:', {k: v.shape for k,v in {'2022':s22,'2023':s23,'2024':s24}.items()})


## 1. Harmonização de nomes de coluna (padrão = a maioria dos anos)

Os três anos nomeiam a mesma informação de formas diferentes. Padronizamos para o nome usado
pela **maioria** (2023/2024). Ex.: 2022 usa `Matem/Portug/Inglês` → vira `Mat/Por/Ing`.

O mapa é **explícito e visível** — cada linha mostra a tradução aplicada, por ano.

In [ ]:
# MAPA DE RENOMEAÇÃO — chave = nome original no ano; valor = nome padronizado
# (padrão adotado = nomenclatura de 2023/2024, que é a maioria)
RENOMEAR_2022 = {
    'Matem':'Mat', 'Portug':'Por', 'Inglês':'Ing',   # notas -> padrão 2023/24
    'Idade 22':'Idade',                               # idade -> padrão
    'Fase ideal':'Fase_Ideal',                        # fase ideal -> padrão
    'Defas':'Defasagem',                              # defasagem -> padrão
    'INDE 22':'INDE_ano',                             # INDE do ano corrente -> nome único
    'Ano nasc':'nasc_raw',                            # nascimento (ano inteiro em 2022)
    'Nome':'Nome_Anon',
}
RENOMEAR_2023 = {
    'Fase Ideal':'Fase_Ideal',
    'INDE 2023':'INDE_ano',
    'Data de Nasc':'nasc_raw',
    'Nome Anonimizado':'Nome_Anon',
}
RENOMEAR_2024 = {
    'Fase Ideal':'Fase_Ideal',
    'INDE 2024':'INDE_ano',
    'Data de Nasc':'nasc_raw',
    'Nome Anonimizado':'Nome_Anon',
}

s22 = s22.rename(columns=RENOMEAR_2022)
s23 = s23.rename(columns=RENOMEAR_2023)
s24 = s24.rename(columns=RENOMEAR_2024)

# IPP não veio como coluna em 2022 -> cria vazia (será tratada na seção 6)
if 'IPP' not in s22.columns:
    s22['IPP'] = np.nan

print('Renomeação aplicada. Conferência dos nomes padronizados presentes:')
for nome, df in [('2022',s22),('2023',s23),('2024',s24)]:
    tem = [c for c in ['Mat','Por','Ing','Fase_Ideal','Defasagem','INDE_ano','nasc_raw'] if c in df.columns]
    print(f'  {nome}: {tem}')


## 2. Fase numérica

Documento (Tabela 4): as fases vão de ALFA (0) a Fase 8 (universitários). Na base, `Fase` aparece
numérica (2022), como texto `FASE N` (2023) e **corrompida** em 2024 (junto da turma, ex. `"1A"`).
Normalizamos para inteiro 0–9 (`ALFA`→0).

In [ ]:
def extrai_fase(v):
    if pd.isna(v): return np.nan
    v = str(v).strip().upper()
    if v.startswith('ALFA'): return 0
    m = re.search(r'(\d+)', v)
    return int(m.group(1)) if m else np.nan

for df in (s22, s23, s24):
    df['fase_num'] = df['Fase'].apply(extrai_fase)
print('Fase_num nula:', s22.fase_num.isna().sum(), s23.fase_num.isna().sum(), s24.fase_num.isna().sum())
print('Fases 2024 (corrigidas):', sorted(s24.fase_num.dropna().unique().tolist()))


## 3. Ano de nascimento na origem (evita erro de tipo misto)

⚠️ 2022 traz **ano inteiro** (`2005`); 2023/24 trazem **data completa**. Extraímos o ano em cada
aba com seu tipo, ANTES de empilhar (senão o pandas leria `2005` como epoch → ano 1970).

In [ ]:
s22['ano_nasc_obs'] = pd.to_numeric(s22['nasc_raw'], errors='coerce')          # já é ano
s23['ano_nasc_obs'] = pd.to_datetime(s23['nasc_raw'], errors='coerce').dt.year
s24['ano_nasc_obs'] = pd.to_datetime(s24['nasc_raw'], errors='coerce').dt.year

_chk = pd.concat([s22.ano_nasc_obs, s23.ano_nasc_obs, s24.ano_nasc_obs]).dropna()
print(f'Range nascimento: {int(_chk.min())}–{int(_chk.max())} | casos 1970 (erro):', int((_chk==1970).sum()))


## 4. Empilhamento em painel único

In [ ]:
for df, ano in [(s22,2022),(s23,2023),(s24,2024)]:
    df['Ano'] = ano

# --- Cria colunas adicionais usando os nomes ORIGINAIS ainda presentes em cada sheet ---
# Pedra do ano corrente (nome difere por ano)
s22['Pedra_ano'] = s22['Pedra 22']
s23['Pedra_ano'] = s23['Pedra 2023']
s24['Pedra_ano'] = s24['Pedra 2024']
# Instituição (recorte social)
for d in (s22,s23,s24):
    d['Instituicao'] = d['Instituição de ensino'] if 'Instituição de ensino' in d.columns else np.nan
# Histórico de Pedra (presença varia)
for d in (s22,s23,s24):
    d['Pedra20']      = d['Pedra 20'] if 'Pedra 20' in d.columns else np.nan
    d['Pedra21']      = d['Pedra 21'] if 'Pedra 21' in d.columns else np.nan
    d['Pedra22_hist'] = d['Pedra 22'] if 'Pedra 22' in d.columns else np.nan
    d['Pedra23_hist'] = d['Pedra 23'] if 'Pedra 23' in d.columns else np.nan
# Histórico de INDE (anos anteriores) - alimenta features de tendência do modelo
for d in (s22,s23,s24):
    d['INDE22_hist'] = pd.to_numeric(d['INDE 22'], errors='coerce') if 'INDE 22' in d.columns else np.nan
    d['INDE23_hist'] = pd.to_numeric(d['INDE 23'], errors='coerce') if 'INDE 23' in d.columns else np.nan
# Descritivas (só 2022 tem)
for d in (s22,s23,s24):
    d['Atingiu_PV']     = d['Atingiu PV'] if 'Atingiu PV' in d.columns else np.nan
    d['Rec_Psicologia'] = d['Rec Psicologia'] if 'Rec Psicologia' in d.columns else np.nan

cols = ['RA','Ano','Fase','fase_num','Fase_Ideal','Defasagem','INDE_ano',
        'IAA','IEG','IPS','IPP','IDA','IPV','IAN','Mat','Por','Ing',
        'Idade','ano_nasc_obs','Gênero','Ano ingresso',
        'Pedra_ano','Instituicao','Pedra20','Pedra21','Pedra22_hist','Pedra23_hist',
        'INDE22_hist','INDE23_hist','Atingiu_PV','Rec_Psicologia']

def seleciona(df):
    d = df.copy()
    for c in cols:
        if c not in d.columns: d[c] = np.nan
    return d[cols]

painel = pd.concat([seleciona(s22), seleciona(s23), seleciona(s24)], ignore_index=True)
for c in ['IAA','IEG','IPS','IPP','IDA','IPV','IAN','Mat','Por','Ing','Defasagem','Idade','INDE_ano','INDE22_hist','INDE23_hist']:
    painel[c] = pd.to_numeric(painel[c], errors='coerce')
painel_original = painel.copy()   # camada 1: base harmonizada, sem nenhuma correção
print('Painel:', painel.shape)
print('Pedra_ano preenchida:', painel['Pedra_ano'].notna().sum())
print('INDE23_hist preenchida:', painel['INDE23_hist'].notna().sum())
print(painel.groupby('Ano').size())

## 5. Ano de nascimento definitivo + idade padronizada

Ano definitivo = **moda** entre os anos observados. 4 casos de divergência genuína resolvidos por
triangulação fase/idade/defasagem (override auditável). Idade = `Ano − ano_nascimento`
(convenção definida em `CONVENCAO_IDADE`).

In [ ]:
override = pd.DataFrame({
    'RA':['RA-766','RA-454','RA-444','RA-178'],
    'ano_nasc':[2012,2008,2011,2009],
    'justificativa':[
        'Fase ideal 2 + idade 11 + defasagem -2->-1 => 2012; Ano nasc 2022 (2011) inconsistente',
        'Anos 2022 e 2023 concordam (2008); 2024 (2007) diverge por digitacao',
        'Anos 2022 e 2023 concordam (2011); 2024 (2009) diverge por digitacao',
        'Anos 2022 e 2023 concordam (2009); 2024 (2008) diverge por digitacao',
    ]})

ano_moda = (painel.dropna(subset=['ano_nasc_obs']).groupby('RA')['ano_nasc_obs']
            .agg(lambda x:int(x.mode().iloc[0])).rename('ano_nasc').reset_index())
ano_final = ano_moda.merge(override[['RA','ano_nasc']], on='RA', how='left', suffixes=('','_ovr'))
n_override = ano_final['ano_nasc_ovr'].notna().sum()
ano_final['ano_nasc'] = ano_final['ano_nasc_ovr'].fillna(ano_final['ano_nasc']).astype(int)
painel = painel.merge(ano_final[['RA','ano_nasc']], on='RA', how='left')

painel['idade_padronizada'] = painel['Ano'] - painel['ano_nasc']   # convenção 31-12
log_impacto['override_nascimento'] = int(n_override)
print('Override aplicado em', n_override, 'alunos. RA-766 =', painel.loc[painel.RA=='RA-766','ano_nasc'].iloc[0])
print('Idade min/max:', int(painel.idade_padronizada.min()), '/', int(painel.idade_padronizada.max()))


## 5b. Fase Ideal e Defasagem — recálculo pela Tabela 4

A `Fase_Ideal` que veio da base diverge da Tabela 4 do documento em ~16,6% das linhas
(confere com a orientação de que a base precisava de tratamento). Como a Defasagem é
`Fase Efetiva − Fase Ideal`, o erro se propaga para a Defasagem e para o alvo do modelo.

Recalculo a Fase Ideal pela régua idade→fase da Tabela 4, usando `idade_padronizada`
(= Ano letivo − ano de nascimento, já com o override aplicado), e refaço a Defasagem.
Guardo as versões originais da base para auditoria.

In [ ]:
# Tabela 4 (idade -> fase ideal). Faixas fecham no limite superior da idade.
def fase_ideal_por_idade(idade):
    if pd.isna(idade): return np.nan
    i = int(idade)
    if i <= 8:  return 0   # 7-8 anos: ALFA
    if i == 9:  return 1   # 3o/4o ano
    if i <= 11: return 2   # 5o/6o
    if i <= 13: return 3   # 7o/8o
    if i == 14: return 4
    if i == 15: return 5
    if i == 16: return 6
    if i == 17: return 7
    return 8               # 18+: universitario

painel['Fase_Ideal_base'] = painel['Fase_Ideal']
painel['Defasagem_base']  = painel['Defasagem']

painel['fase_ideal_num'] = painel['idade_padronizada'].apply(fase_ideal_por_idade)
painel['Fase_Ideal'] = painel['fase_ideal_num']
painel['Defasagem']  = painel['fase_num'] - painel['fase_ideal_num']

n_mudou = int((painel['Defasagem'] != painel['Defasagem_base']).sum())
print(f'Fase Ideal recalculada pela Tabela 4. Defasagem alterada em {n_mudou} linhas.')
print('Defasagem min/max apos recalculo:', int(painel['Defasagem'].min()), '/', int(painel['Defasagem'].max()))
log_impacto['defasagem_recalculada'] = n_mudou

# IAN e funcao-degrau da Defasagem (Tabela 41). Como a Defasagem mudou, refaco o IAN
# onde ele existe, para manter a conformidade com o documento.
def ian_tabela41(d):
    if pd.isna(d): return np.nan
    if d >= 0:  return 10.0
    if d >= -2: return 5.0
    return 2.5

painel['IAN_base'] = painel['IAN']
tem_ian = painel['IAN'].notna()
painel.loc[tem_ian, 'IAN'] = painel.loc[tem_ian, 'Defasagem'].apply(ian_tabela41)
print('IAN recalculado pela Tabela 41 em', int(tem_ian.sum()), 'linhas.')

## 5c. Progressão de fase — correção do além-do-ideal e movimentos atípicos

A fase efetiva é dado administrativo e nenhuma fonte do projeto documenta a mecânica de progressão. Três situações nos 1.365 pares consecutivos: (1) **19 casos além do ideal** (salto ≥ 2 que deixa o aluno *acima* da fase da própria idade — ex.: 15–16 anos em Fase 7 com ideal 5; validados um a um) → corrigidos para `fase do ano anterior + 1`, iterativo por ano, com `fase_override.csv` prevalecendo linha a linha; (2) **saltos ≥ 2 em direção à fase da idade** e (3) **regressões** → sem régua para corrigir; tratados conforme `TRATAMENTO_SALTOS_COERENTES` (padrão: excluídos da base de modelo por conservadorismo, permanecendo no painel limpo; reavaliar após o plantão).

In [ ]:
painel['fase_num_base'] = painel['fase_num']

# override manual tem precedência sobre a regra (mesmo mecanismo do nasc_override)
import os
_ovr_path = PASTA_SAIDA + 'fase_override.csv'
if os.path.exists(_ovr_path):
    fase_override = pd.read_csv(_ovr_path)
else:
    fase_override = pd.DataFrame(columns=['RA','Ano','fase_corrigida'])
    os.makedirs(PASTA_SAIDA, exist_ok=True)
    fase_override.to_csv(_ovr_path, index=False)   # template para ajustes pós-plantão

n_corr = 0
if APLICAR_CORRECAO_FASE:
    fmap = {(r, a): f for r, a, f in zip(painel['RA'], painel['Ano'], painel['fase_num'])}
    for ano in (2023, 2024):
        m = painel['Ano'].eq(ano)
        for i in painel.index[m]:
            ra = painel.at[i, 'RA']
            f_atual, f_ideal = painel.at[i, 'fase_num'], painel.at[i, 'fase_ideal_num']
            f_ant = fmap.get((ra, ano - 1), np.nan)
            if pd.notna(f_ant) and pd.notna(f_atual) and pd.notna(f_ideal) \
               and (f_atual - f_ant >= 2) and (f_atual > f_ideal):
                painel.at[i, 'fase_num'] = f_ant + 1
                fmap[(ra, ano)] = f_ant + 1
                n_corr += 1
for _, r in fase_override.iterrows():
    m = (painel['RA'] == r['RA']) & (painel['Ano'] == r['Ano'])
    painel.loc[m, 'fase_num'] = r['fase_corrigida']
log_impacto['correcao_fase'] = n_corr
print(f'Fase corrigida (além do ideal, regra fase_anterior+1): {n_corr} registros | overrides: {len(fase_override)}')

# a fase mudou -> Defasagem e IAN refeitos (mesmas regras da seção 5b)
painel['Defasagem'] = painel['fase_num'] - painel['fase_ideal_num']
tem_ian = painel['IAN_base'].notna()
painel.loc[tem_ian, 'IAN'] = painel.loc[tem_ian, 'Defasagem'].apply(ian_tabela41)

# movimentos atípicos remanescentes: salto >= 2 (rumo ao ideal ou p/ Fase 8) ou regressão.
# Marco as DUAS pontas do par; a exclusão do modelo acontece na seção 9.
painel['_movimento_fase_atipico'] = False
_b = painel.dropna(subset=['fase_num'])[['RA','Ano','fase_num']]
_p = _b.merge(_b, on='RA', suffixes=('_T','_T1'))
_p = _p[_p['Ano_T1'] == _p['Ano_T'] + 1]
_p['delta'] = _p['fase_num_T1'] - _p['fase_num_T']
_at = _p[(_p['delta'] >= 2) | (_p['delta'] < 0)]
pontas = set(zip(_at['RA'], _at['Ano_T'])) | set(zip(_at['RA'], _at['Ano_T1']))
painel.loc[[k in pontas for k in zip(painel['RA'], painel['Ano'])], '_movimento_fase_atipico'] = True
log_impacto['saltos_atipicos_pares'] = int(len(_at))
log_impacto['saltos_atipicos_linhas'] = int(painel['_movimento_fase_atipico'].sum())
print(f'Movimentos atípicos pós-correção: {len(_at)} pares ({int((_at.delta>=2).sum())} saltos, '
      f'{int((_at.delta<0).sum())} regressões) -> {int(painel._movimento_fase_atipico.sum())} linhas marcadas')

## 6. IDA — recálculo conforme o documento

**Documento:** `IDA = (Nota Matemática + Nota Português + Nota Inglês) / 3`.
A base mistura arredondamentos → recalculamos. Inglês não é avaliado nas Fases 0-2, então
`IDA_MEDIA_NOTAS_PRESENTES=True` usa a média das notas que existem (2 nessas fases).

In [ ]:
painel['IDA_original'] = painel['IDA']

if RECALCULAR_IDA:
    def calcula_ida(row):
        notas = [x for x in (row['Mat'], row['Por'], row['Ing']) if pd.notna(x)]
        if IDA_MEDIA_NOTAS_PRESENTES:
            return round(sum(notas)/len(notas), 1) if notas else np.nan
        else:
            if pd.notna(row['Mat']) and pd.notna(row['Por']) and pd.notna(row['Ing']):
                return round((row['Mat']+row['Por']+row['Ing'])/3, 1)
            return np.nan
    painel['IDA'] = painel.apply(calcula_ida, axis=1)
    mudou = (painel['IDA'] != painel['IDA_original']).sum()
    log_impacto['IDA_recalculado'] = int(mudou)
    print(f'IDA recalculado. Linhas alteradas vs original: {mudou}. Nulos: {painel.IDA.isna().sum()}')
else:
    print('IDA mantido como veio na base.')


## 7. Pisos de validade e prova de contaminação do INDE

**IAA** (Tabela 40 + Figura 10): piso 3,5 nas Fases 0–2 e 2,5 nas Fases 3–8 — valores abaixo são impossíveis pela escala. Para cada IAA inválido, verifico se o INDE da base bate com a fórmula documental *usando o próprio valor inválido* (âncora no IAN da defasagem original e no IDA original): se bate, a contaminação está provada e IAA/INDE/Pedra da linha são anulados; se a checagem é impossível (2022 sem IPP na base, ou outro componente ausente), o desfecho é o mesmo por indeterminação; se a equação só fecha com um IAA válido (≥ piso), o valor é recuperado. **IPS/IPP/IPV**: zero → NaN por fundamento *empírico* (piso observado ≈ 2,5; nenhum valor entre 0 e o piso na base) — o documento não traz tabela de pontuação para esses três.

In [ ]:
# ---- IAA: piso por fase + prova de contaminação do INDE (antes de qualquer NaN) ----
piso_iaa = np.where(painel['fase_num'] <= 2, PISO_IAA_FASE_0_2, PISO_IAA_FASE_3_8)
painel['_iaa_invalido'] = painel['IAA'].notna() & (painel['IAA'] < piso_iaa)
painel['IAA_registrado_inv'] = np.where(painel['_iaa_invalido'], painel['IAA'], np.nan)

# fórmula com o que a ONG usou: IAN da defasagem original, IDA original, fase registrada
_ian_ong = painel['Defasagem_base'].apply(
    lambda d: np.nan if pd.isna(d) else (10.0 if d >= 0 else (5.0 if d >= -2 else 2.5)))
_f8 = painel['fase_num_base'] == 8
_inde_f = np.where(_f8,
    _ian_ong*0.1 + painel['IDA_original']*0.4 + painel['IEG']*0.2 + painel['IAA']*0.1 + painel['IPS']*0.2,
    _ian_ong*0.1 + painel['IDA_original']*0.2 + painel['IEG']*0.2 + painel['IAA']*0.1
    + painel['IPS']*0.1 + painel['IPP']*0.1 + painel['IPV']*0.2)
_delta = painel['INDE_ano'] - _inde_f
_checavel = painel['_iaa_invalido'] & pd.Series(_inde_f, index=painel.index).notna() & painel['INDE_ano'].notna()
_iaa_impl = painel['IAA'] + _delta/0.1     # IAA que fecharia a equação exatamente

m_prova   = _checavel & (_delta.abs() <= TOL_CONTAMINACAO)
m_recup   = _checavel & ~m_prova & (_iaa_impl >= piso_iaa) & (_iaa_impl <= 10 + 1e-6)
m_indet   = painel['_iaa_invalido'] & ~m_prova & ~m_recup
painel['_inde_contaminado'] = m_prova | m_indet

painel.loc[m_recup, 'IAA'] = _iaa_impl[m_recup].round(3)          # nenhum caso na base atual; trava fica
painel.loc[m_prova | m_indet, 'IAA'] = np.nan
log_impacto['iaa_invalido'] = int(painel['_iaa_invalido'].sum())
log_impacto['inde_contaminacao_provada'] = int(m_prova.sum())
log_impacto['inde_indeterminado'] = int(m_indet.sum())
print(f"IAA abaixo do piso da fase: {int(painel._iaa_invalido.sum())} "
      f"(prova de contaminação: {int(m_prova.sum())} | indeterminados: {int(m_indet.sum())} "
      f"| recuperados por equação: {int(m_recup.sum())})")

if TRATAR_ZEROS_IMPOSSIVEIS:
    total = 0
    for col in ['IPS','IPP','IPV']:                # IAA tem regra própria acima
        n0 = int((painel[col]==0).sum())
        painel.loc[painel[col]==0, col] = np.nan
        total += n0
        print(f'  {col}: {n0} zeros -> NaN (piso empírico)')
    log_impacto['zeros_impossiveis'] = total

if TRATAR_IEG_ZERO_NO_CICLO_VAZIO:
    cc = ['IAA','IPS','IPP','IDA','IPV','Mat','Por']
    m = (painel['IEG']==0) & (painel[cc].isna().sum(axis=1) >= 5)
    print(f'  IEG: {int(m.sum())} zeros no ciclo vazio -> NaN')
    painel.loc[m, 'IEG'] = np.nan
    log_impacto['IEG_zero_ciclo'] = int(m.sum())


## 8. Reconstrução do IPP de 2022 (engenharia reversa do INDE)

**Documento (composição do INDE, Fases 0-7):**
`INDE = IAN·0,1 + IDA·0,2 + IEG·0,2 + IAA·0,1 + IPS·0,1 + IPP·0,1 + IPV·0,2`
O IPP **não veio como coluna em 2022, mas compõe o INDE** → isolamos algebricamente.
Só onde os outros 6 índices estão íntegros. Fórmula da Fase 8 (sem IPP) não se aplica: não há Fase 8 em 2022.

In [ ]:
painel['IPP_origem'] = np.where(painel['IPP'].notna(), 'coletado', 'ausente')

if RECONSTRUIR_IPP_2022:
    mask22 = (painel['Ano']==2022)
    outros6 = ['IAN','IDA','IEG','IAA','IPS','IPV']
    limpo = mask22 & painel['INDE_ano'].notna()
    for c in outros6:
        limpo &= painel[c].notna() & (painel[c]!=0)
    # O INDE da base foi calculado pela ONG com o IAN da defasagem original.
    # Para extrair o IPP correto, a conta precisa usar esse mesmo IAN.
    _ian_inde = painel['Defasagem_base'].apply(
        lambda d: np.nan if pd.isna(d) else (10.0 if d >= 0 else (5.0 if d >= -2 else 2.5)))
    resto = (_ian_inde*0.1 + painel['IDA']*0.2 + painel['IEG']*0.2 +
             painel['IAA']*0.1 + painel['IPS']*0.1 + painel['IPV']*0.2)
    painel.loc[limpo, 'IPP'] = ((painel['INDE_ano']-resto)/0.1)[limpo].round(3)
    painel.loc[limpo, 'IPP_origem'] = 'reconstruido_2022'
    log_impacto['IPP_reconstruido'] = int(limpo.sum())

    med_rec = painel.loc[painel.IPP_origem=='reconstruido_2022','IPP'].mean()
    med_col = painel.loc[painel.IPP_origem=='coletado','IPP'].mean()
    print(f'IPP reconstruído em {int(limpo.sum())} casos (2022).')
    print(f'  média reconstruída: {med_rec:.2f} | coletada 2023-24: {med_col:.2f}  (dif ~{med_col-med_rec:.1f} pt: registrar no relatório)')

if CLIP_INDICES_FORA_FAIXA:
    # Documento: escala 0-10. Valores levemente acima (ex: 10,01 por arredondamento) -> ajusta para 10.
    # Só valores GENUINAMENTE inválidos (negativos, ou > 10,5) viram NaN.
    TOL = 0.1  # tolerância de arredondamento acima de 10 (conservador)
    n_ajust, n_nan = 0, 0
    for col in ['IAA','IEG','IPS','IPP','IPV','IDA']:
        # IPP reconstruído é tratado à parte (decisão: excluir a linha, não zerar) -> não mexer aqui
        eh_ipp_rec = (painel['IPP_origem']=='reconstruido_2022') if col=='IPP' else pd.Series(False, index=painel.index)
        # arredonda para 10 os que estão em (10, 10+TOL]
        m_ajusta = (painel[col] > 10) & (painel[col] <= 10 + TOL) & (~eh_ipp_rec)
        n_ajust += int(m_ajusta.sum())
        painel.loc[m_ajusta, col] = 10.0
        # NaN para os genuinamente inválidos (exceto IPP reconstruído)
        m_invalido = ((painel[col] < 0) | (painel[col] > 10 + TOL)) & (~eh_ipp_rec)
        n_nan += int(m_invalido.sum())
        painel.loc[m_invalido, col] = np.nan
    log_impacto['clip_ajustado_p10'] = n_ajust
    log_impacto['clip_invalido_nan'] = n_nan
    print(f'Valores levemente acima de 10 (arredondamento) -> ajustados p/ 10: {n_ajust}')
    print(f'Valores genuinamente inválidos (<0 ou >10,1) -> NaN: {n_nan}')


# IPP reconstruído fora da faixa válida [0,10] -> linha marcada para exclusão (imaterialidade)
mask_ipp_invalido = (painel['IPP_origem']=='reconstruido_2022') & ((painel['IPP']<0)|(painel['IPP']>10))
painel.loc[mask_ipp_invalido, 'IPP'] = np.nan
painel['_ipp_reconstruido_invalido'] = mask_ipp_invalido
log_impacto['IPP_reconstruido_invalido_excluir'] = int(mask_ipp_invalido.sum())
print(f'IPP reconstruído inválido (fora de [0,10]) -> linha marcada p/ exclusão: {int(mask_ipp_invalido.sum())}')

## 8b. Recuperação algébrica de índices isolados (generalização do IPP)

Estende a lógica do IPP-reconstruído para **qualquer** índice do INDE que falte sozinho numa linha.

**Princípio:** se uma linha tem o INDE preenchido e **exatamente 1** dos 7 índices ausente,
esse índice é recuperável **exatamente** pela fórmula do INDE (1 equação, 1 incógnita) —
valor real, melhor que a estimativa do KNN.

**Salvaguarda:** só aceita o valor recuperado se cair na faixa válida [0,10]. Alguns casos dão
valor levemente negativo (~-0,08) — isso ocorre quando a soma ponderada dos outros índices já
excede o INDE por arredondamento; nesses casos o IAA real seria ~0, que é **impossível** pela
régua do questionário (piso 2,5/3,5). Esses casos são rejeitados e vão para o KNN.

Resultado: ~152 índices recuperados com valor exato (majoritariamente IAA). Casos com 2+ ausentes
(sistema indeterminado, ex. RA-776) ou fora de faixa ficam para o KNN na fase de modelagem.

In [ ]:
RECUPERAR_INDICES_ALGEBRICO = True  # recupera índice isolado ausente via fórmula do INDE?

if RECUPERAR_INDICES_ALGEBRICO:
    # pesos do INDE (Fases 0-7). IPP já foi reconstruído na etapa anterior onde possível.
    PESOS = {'IAN':0.1,'IDA':0.2,'IEG':0.2,'IAA':0.1,'IPS':0.1,'IPP':0.1,'IPV':0.2}
    idx_inde = list(PESOS.keys())

    # só fases 0-7 (fórmula com 7 índices); Fase 8 tem fórmula diferente e é excluída do modelo
    mask_07 = ~painel['fase_num'].isin([8,9])
    n_ausentes = painel[idx_inde].isna().sum(axis=1)
    # candidatos: INDE presente, exatamente 1 índice ausente, fase 0-7
    cand = mask_07 & painel['INDE_ano'].notna() & (n_ausentes == 1) & ~painel['_inde_contaminado']

    recuperados = 0
    painel['_recuperado_algebrico'] = ''
    for idx in idx_inde:
        # linhas candidatas onde ESTE índice é o ausente
        m = cand & painel[idx].isna()
        if m.sum() == 0:
            continue
        # soma ponderada dos OUTROS índices (todos presentes nessas linhas)
        outros = [c for c in idx_inde if c != idx]
        # IAN na soma = o da defasagem original (e o que esta dentro do INDE da base)
        _ian_inde = painel['Defasagem_base'].apply(
            lambda d: np.nan if pd.isna(d) else (10.0 if d >= 0 else (5.0 if d >= -2 else 2.5)))
        soma_outros = sum((_ian_inde.loc[m] if c == 'IAN' else painel.loc[m, c])*PESOS[c]
                          for c in outros)
        valor = ((painel.loc[m,'INDE_ano'] - soma_outros) / PESOS[idx]).round(3)
        # só aceita se cair na faixa válida: [piso do índice, 10]
        piso_col = (pd.Series(np.where(painel.loc[m,'fase_num']<=2, PISO_IAA_FASE_0_2, PISO_IAA_FASE_3_8),
                              index=valor.index) if idx=='IAA'
                    else pd.Series(2.5 if idx in ('IPS','IPP','IPV') else 0.0, index=valor.index))
        valido = (valor >= piso_col) & (valor <= 10)
        painel.loc[m[m].index[valido.values], idx] = valor[valido.values]
        painel.loc[m[m].index[valido.values], '_recuperado_algebrico'] = idx
        recuperados += int(valido.sum())
        print(f'  {idx}: {int(valido.sum())} recuperados por álgebra')
    log_impacto['recuperado_algebrico'] = recuperados
    print(f'Total recuperado por álgebra (valor exato, não estimado): {recuperados}')


## 8b. INDE e Pedra — recálculo pós-correções

O INDE da base foi calculado pela ONG com o IAN antigo (defasagem pré-correção). Como o IAN
mudou em ~9% das linhas, recalculo o INDE pela fórmula oficial com os índices já corrigidos
e derivo a Pedra pelos cortes do documento (Quartzo <6,1 · Ágata <7,2 · Ametista <8,2 · Topázio).
Os valores originais ficam em colunas *_base para auditoria. Fase 8 usa a fórmula própria
(sem IPP/IPV).

In [ ]:
painel['INDE_ano_base'] = painel['INDE_ano']
painel['Pedra_ano_base'] = painel['Pedra_ano']

f07 = ~painel['fase_num'].isin([8, 9])
f8  = painel['fase_num'] == 8

inde07 = (painel['IAN']*0.1 + painel['IDA']*0.2 + painel['IEG']*0.2 + painel['IAA']*0.1
          + painel['IPS']*0.1 + painel['IPP']*0.1 + painel['IPV']*0.2)
inde8  = (painel['IAN']*0.1 + painel['IDA']*0.4 + painel['IEG']*0.2 + painel['IAA']*0.1
          + painel['IPS']*0.2)
painel.loc[f07, 'INDE_ano'] = inde07[f07].round(6)
painel.loc[f8,  'INDE_ano'] = inde8[f8].round(6)

def pedra_do_inde(v):
    if pd.isna(v): return np.nan
    if v < 6.110: return 'Quartzo'    # cortes exatos das Figuras 2-3 (média 7,154, sigma 1,044)
    if v < 7.154: return 'Ágata'
    if v < 8.198: return 'Ametista'
    return 'Topázio'
painel['Pedra_ano'] = painel['INDE_ano'].apply(pedra_do_inde)

dif = (painel['INDE_ano'] - painel['INDE_ano_base']).abs()
n_inde = int((dif > 0.01).sum())
_tem_base = painel['Pedra_ano_base'].notna()
n_reclass = int(((painel['Pedra_ano'] != painel['Pedra_ano_base']) & painel['Pedra_ano'].notna() & _tem_base).sum())
n_anulada = int((painel['Pedra_ano'].isna() & _tem_base).sum())
print(f'INDE recalculado; difere da base em {n_inde} linhas.')
print(f'Pedra pelos cortes exatos (6,110/7,154/8,198): {n_reclass} reclassificadas | '
      f'{n_anulada} anuladas (INDE não recalculável: componente ausente ou contaminado)')
log_impacto['pedra_anulada'] = n_anulada
log_impacto['inde_recalculado'] = n_inde
log_impacto['pedra_rederivada'] = n_reclass

# Historicos (INDE22/23_hist, Pedra22/23_hist) vieram da base com o INDE antigo.
# Onde o aluno esta no painel no ano de referencia, atualizo com o valor recalculado.
_atual = 0
for ano_h, col_i, col_p in [(2022, 'INDE22_hist', 'Pedra22_hist'),
                            (2023, 'INDE23_hist', 'Pedra23_hist')]:
    ref = painel[painel['Ano'] == ano_h].set_index('RA')[['INDE_ano', 'Pedra_ano']]
    m = (painel['Ano'] > ano_h) & painel['RA'].isin(ref.index) & painel[col_i].notna()
    painel.loc[m, col_i] = painel.loc[m, 'RA'].map(ref['INDE_ano'])
    painel.loc[m, col_p] = painel.loc[m, 'RA'].map(ref['Pedra_ano'])
    _atual += int(m.sum())
print(f'Historicos INDE/Pedra atualizados a partir do painel recalculado: {_atual} valores.')
log_impacto['historicos_atualizados'] = _atual

## 9. Flags de exclusão (parametrizadas)

**7 índices do INDE** (documento) ≠ **5 features do modelo**. Saem das features:
`IAN` (função-degrau da Defasagem = alvo → vazamento) e `IPP` (natureza mista;
controlado por `IPP_COMO_FEATURE`). Critério de exclusão usa `CORTE_MIN_FEATURES_MODELO`.

In [ ]:
painel['excluir_tudo'] = (painel['fase_num']==9) & EXCLUIR_FASE_9_DE_TUDO

FEATURES_MODELO = ['IAA','IEG','IPS','IDA','IPV']
if IPP_COMO_FEATURE:
    FEATURES_MODELO = FEATURES_MODELO + ['IPP']
painel['n_feat_modelo'] = painel[FEATURES_MODELO].notna().sum(axis=1)

painel['motivo_exclusao_modelo'] = ''
if EXCLUIR_FASE_9_DE_TUDO:
    painel.loc[painel.fase_num==9, 'motivo_exclusao_modelo'] = 'matricula_em_processamento'
if EXCLUIR_FASE_8_DO_MODELO:
    painel.loc[painel.fase_num==8, 'motivo_exclusao_modelo'] = 'universitario_sem_coleta'
if TRATAMENTO_SALTOS_COERENTES == 'excluir':
    m_at = painel['_movimento_fase_atipico'] & (painel.motivo_exclusao_modelo=='')
    painel.loc[m_at, 'motivo_exclusao_modelo'] = 'movimento_fase_atipico'
m_poucas = (painel.motivo_exclusao_modelo=='') & (painel.n_feat_modelo < CORTE_MIN_FEATURES_MODELO)
painel.loc[m_poucas, 'motivo_exclusao_modelo'] = 'poucas_features_modelo'
# IPP reconstruído inválido também sai do modelo (decisão: descartar por imaterialidade, não zerar)
if '_ipp_reconstruido_invalido' in painel.columns:
    painel.loc[painel['_ipp_reconstruido_invalido'] & (painel.motivo_exclusao_modelo==''), 'motivo_exclusao_modelo'] = 'ipp_reconstruido_invalido'

# ZERO IMPUTAÇÃO: feature ainda ausente após recuperação algébrica => exclui (sem KNN)
if ZERO_IMPUTACAO:
    m_nan = (painel['motivo_exclusao_modelo']=='') & (painel[FEATURES_MODELO].isna().any(axis=1))
    painel.loc[m_nan, 'motivo_exclusao_modelo'] = 'feature_ausente_sem_imputacao'

painel['excluir_modelo'] = painel.motivo_exclusao_modelo != ''

# NOTA: Fase 8 (universitários) sai do MODELO (sem features preditoras: 0/127),
# mas permanece no painel_limpo para ANÁLISE DESCRITIVA SEPARADA (excluir_tudo=False).

log_impacto['excluir_tudo'] = int(painel.excluir_tudo.sum())
log_impacto['excluir_modelo'] = int(painel.excluir_modelo.sum())
print(painel['motivo_exclusao_modelo'].replace('','(mantido)').value_counts())


## 10. Verificações de conformidade (base vs documento)

In [ ]:
print('='*55); print('RESUMO'); print('='*55)
base_mod = painel[~painel.excluir_tudo & ~painel.excluir_modelo]
print(f'Total: {len(painel)} | sem fabricados: {(~painel.excluir_tudo).sum()} | MODELAGEM: {len(base_mod)}')

# Conformidade com o documento (Tabela 41): IAN é função-degrau de 3 valores
chk = painel.dropna(subset=['IAN','Defasagem'])
regra = np.where(chk.Defasagem>=0,10.0,np.where(chk.Defasagem>=-2,5.0,2.5))
print(f'\n[Tabela 41] IAN respeita a regra de 3 valores: {(np.isclose(chk.IAN,regra)).mean()*100:.1f}% (deve ser 100%)')
print(f'[Tabela 40] Zeros restantes em IAA/IPS/IPP/IPV: {{c:int((painel[c]==0).sum()) for c in ["IAA","IPS","IPP","IPV"]}}'.replace('{c:int((painel[c]==0).sum()) for c in ["IAA","IPS","IPP","IPV"]}', str({c:int((painel[c]==0).sum()) for c in ['IAA','IPS','IPP','IPV']})))
print(f'[faixa 0-10] valores fora de faixa: {{c:int(((painel[c]<0)|(painel[c]>10)).sum()) for c in ["IDA","IEG","IAA","IPS","IPP","IPV"]}}'.replace('{c:int(((painel[c]<0)|(painel[c]>10)).sum()) for c in ["IDA","IEG","IAA","IPS","IPP","IPV"]}', str({c:int(((painel[c]<0)|(painel[c]>10)).sum()) for c in ['IDA','IEG','IAA','IPS','IPP','IPV']})))
_piso_chk = np.where(painel['fase_num']<=2, PISO_IAA_FASE_0_2, PISO_IAA_FASE_3_8)
print(f'[Tabela 40] IAA abaixo do piso da fase restante: {int((painel.IAA < _piso_chk).sum())} (deve ser 0)')
print(f'[contaminação] INDE/Pedra anulados por IAA inválido: {int(painel._inde_contaminado.sum())}')
print(f'[idade] min/max base modelo: {int(base_mod.idade_padronizada.min())}/{int(base_mod.idade_padronizada.max())}')
assert base_mod.idade_padronizada.max() < 30, 'ERRO: idade implausível'

print('\nTaxa Defasagem<=-1 por ano (%):',
      (base_mod.assign(r=base_mod.Defasagem<=-1).groupby('Ano').r.mean().round(3)*100).to_dict())
print('% ausência nas features:', (base_mod[FEATURES_MODELO].isna().mean()*100).round(1).to_dict())


## 11. Tabela de impacto das decisões

Quantas linhas/valores cada decisão do PAINEL afetou — para auditoria de relance.

In [ ]:
tabela = pd.DataFrame([
    {'decisão':'Override manual de nascimento', 'parâmetro':'(4 casos fixos)', 'linhas/valores afetados':log_impacto.get('override_nascimento',0)},
    {'decisão':'IDA recalculado', 'parâmetro':f'RECALCULAR_IDA={RECALCULAR_IDA}', 'linhas/valores afetados':log_impacto.get('IDA_recalculado',0)},
    {'decisão':'Zeros impossíveis -> NaN', 'parâmetro':f'TRATAR_ZEROS_IMPOSSIVEIS={TRATAR_ZEROS_IMPOSSIVEIS}', 'linhas/valores afetados':log_impacto.get('zeros_impossiveis',0)},
    {'decisão':'IEG zero em ciclo vazio -> NaN', 'parâmetro':f'TRATAR_IEG_ZERO_NO_CICLO_VAZIO={TRATAR_IEG_ZERO_NO_CICLO_VAZIO}', 'linhas/valores afetados':log_impacto.get('IEG_zero_ciclo',0)},
    {'decisão':'Fase corrigida (além do ideal)', 'parâmetro':f'APLICAR_CORRECAO_FASE={APLICAR_CORRECAO_FASE}', 'linhas/valores afetados':log_impacto.get('correcao_fase',0)},
    {'decisão':'Movimento de fase atípico -> fora do modelo', 'parâmetro':f"TRATAMENTO_SALTOS_COERENTES='{TRATAMENTO_SALTOS_COERENTES}'", 'linhas/valores afetados':log_impacto.get('saltos_atipicos_linhas',0)},
    {'decisão':'IAA abaixo do piso da fase', 'parâmetro':f'pisos {PISO_IAA_FASE_0_2}/{PISO_IAA_FASE_3_8}', 'linhas/valores afetados':log_impacto.get('iaa_invalido',0)},
    {'decisão':'INDE contaminado (prova + indeterminados)', 'parâmetro':f'TOL_CONTAMINACAO={TOL_CONTAMINACAO}', 'linhas/valores afetados':log_impacto.get('inde_contaminacao_provada',0)+log_impacto.get('inde_indeterminado',0)},
    {'decisão':'IPP 2022 reconstruído', 'parâmetro':f'RECONSTRUIR_IPP_2022={RECONSTRUIR_IPP_2022}', 'linhas/valores afetados':log_impacto.get('IPP_reconstruido',0)},
    {'decisão':'Valores ~10 (arred.) ajustados p/ 10', 'parâmetro':f'CLIP_INDICES_FORA_FAIXA={CLIP_INDICES_FORA_FAIXA}', 'linhas/valores afetados':log_impacto.get('clip_ajustado_p10',0)},
    {'decisão':'Valores inválidos (<0 ou >10,5) -> NaN', 'parâmetro':f'CLIP_INDICES_FORA_FAIXA={CLIP_INDICES_FORA_FAIXA}', 'linhas/valores afetados':log_impacto.get('clip_invalido_nan',0)},
    {'decisão':'Excluídos de tudo (fabricados)', 'parâmetro':f'EXCLUIR_FASE_9_DE_TUDO={EXCLUIR_FASE_9_DE_TUDO}', 'linhas/valores afetados':log_impacto.get('excluir_tudo',0)},
    {'decisão':'Excluídos do modelo', 'parâmetro':f'CORTE_MIN_FEATURES={CORTE_MIN_FEATURES_MODELO}', 'linhas/valores afetados':log_impacto.get('excluir_modelo',0)},
])
print(tabela.to_string(index=False))


## 12. Exportação

In [ ]:
import os
os.makedirs(PASTA_SAIDA, exist_ok=True)
painel_original.to_csv(PASTA_SAIDA+'painel_original.csv', index=False, encoding='utf-8-sig')  # camada 1
painel.to_csv(PASTA_SAIDA+'painel_limpo.csv', index=False, encoding='utf-8-sig')
base_mod = painel[~painel.excluir_tudo & ~painel.excluir_modelo].copy()
base_mod.to_csv(PASTA_SAIDA+'painel_modelo.csv', index=False, encoding='utf-8-sig')
override.to_csv(PASTA_SAIDA+'nasc_override.csv', index=False, encoding='utf-8-sig')
print('Salvos:', PASTA_SAIDA)
print(f'  painel_original.csv -> {painel_original.shape}')
print(f'  painel_limpo.csv  -> {painel.shape}')
print(f'  painel_modelo.csv -> {base_mod.shape}')
print('\nBase pronta. ✅')
